# Week 04 Homework Submission

**Mini-project goal:** 

* Build a simple SQL agent that queries NYC taxi data using DuckDB, then write tests for it.
* The first two questions set up the agent. 
* Questions 3 through 6 focus on testing: writing tests, verifying tool calls, using LLM judges, and tracking costs.

## Q1. Set up DuckDB

In [1]:
from sql_tools import setup_database
count = setup_database()

Loaded 2964624 rows


## Q2. Create the Agent

Implement the tools and the agent

### SQLTools Class
Add a SQLTools class to ***sql_tools.py*** with two methods (follow the same pattern we used in the module):
* get_schema() - runs DESCRIBE trips and returns all column names with their types
* run_sql(query) - executes a SQL query and returns results as text (column headers + data rows, limited to 50 rows)


### SQL Agent
Create a separate file called ***sql_agent.py*** with:
* A SQLResult pydantic model with three fields: sql_query, result_text, row_count
* A PydanticAI Agent using gpt-4o-mini, SQLResult as the output type, and the two SQL tools (pass the methods as a list: tools=[sql_tools.get_schema, sql_tools.run_sql])
* Instructions that tell the agent to always start by getting the schema before running queries

In [2]:
import duckdb

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
PARQUET_FILE = "yellow_tripdata_2024-01.parquet"

con = duckdb.connect("taxi.db")

In [3]:
from sql_agent import create_agent, SQLAgentConfig
from sql_tools import SQLTools

sql_agent = create_agent(SQLAgentConfig, search_tools=SQLTools(connection=con))

In [4]:
from sql_agent import run_agent
prompt = "What's the average trip distance for rides with 2 passengers?"
query_result = await run_agent(sql_agent, prompt)

TOOL CALL (query): get_schema({})
TOOL CALL (query): run_sql({"query":"SELECT AVG(trip_distance) AS average_trip_distance FROM trips WHERE passenger_count = 2;"})


Test your agent by running uv run python sql_agent.py and asking: "What's the average trip distance for rides with 2 passengers?"

What's the average trip distance?

In [10]:
print(query_result.output.result_text)


average_trip_distance
3.7827640377879166


## Q3. Write your first test

Create a file called test_agent.py. Write a test that asks the agent "How many trips had more than 5 passengers?" and asserts that:
* output.sql_query is a non-empty string
* output.result_text contains the actual count

Since the data doesn't change, you can verify the exact number. Run the query directly in DuckDB first to find the answer, then assert that the agent's result_text contains that number.
* How many trips had more than 5 passengers?

In [ ]:
%%bash
cd /Users/wesley/workspace/learning/ai-engineering-buildcamp
uv run pytest 04-testing/homework/tests/test_agent.py::test_agent_runs -v -s

============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.3, pluggy-1.6.0 -- /Users/wesley/workspace/learning/ai-engineering-buildcamp/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wesley/workspace/learning/ai-engineering-buildcamp
configfile: pyproject.toml
plugins: asyncio-1.3.0, logfire-4.32.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 1 item

04-testing/homework/tests/test_agent.py::test_agent_runs Loaded 2964624 rows
USER PROMPT (query): How many trips had more than 5 passengers
TOOL CALL (query): get_schema({})
TOOL CALL (query): run_sql({"query":"SELECT COUNT(*) AS trips_with_more_than_5_passengers FROM trips WHERE passenger_count > 5"})

result_text: trips_with_more_than_5_passengers
22413
PASSED

============================== 1 passed in 3.05s ===============================


## Q4. Testing tool calls

Agent should always get the schema first, then run SQL queries. Write a test that verifies this behavior by checking the order of tool calls

Create a file called utils.py with the collect_tools helper from the module. This function extracts tool calls from the agent's message history.

In test_agent.py, write a test that:
* Ask "What is the most common payment type?"
* Asserts the first tool call is get_schema
* Asserts that run_sql is also called


What is the name of the second tool the agent calls?
* get_schema
* run_sql
* describe_table
* list_columns

In [2]:
%%bash
cd /Users/wesley/workspace/learning/ai-engineering-buildcamp

uv run pytest 04-testing/homework/tests/test_agent.py::test_agent_tool_call_order -v -s


============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.3, pluggy-1.6.0 -- /Users/wesley/workspace/learning/ai-engineering-buildcamp/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wesley/workspace/learning/ai-engineering-buildcamp
configfile: pyproject.toml
plugins: asyncio-1.3.0, logfire-4.32.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 1 item

04-testing/homework/tests/test_agent.py::test_agent_tool_call_order Loaded 2964624 rows
USER PROMPT (query): What is the most common payment type?
TOOL CALL (query): get_schema({})
TOOL CALL (query): run_sql({"query":"SELECT payment_type, COUNT(*) as count FROM trips GROUP BY payment_type ORDER BY count DESC LIMIT 1;"})
PASSED

============================== 1 passed in 3.24s ===============================


## Q5. LLM judge test

Add an LLM Judge that evaluates the agent's output using natural language criteria.

Create a judge.py file with the judge from the module(the evaluate_agent_performance and assert_criteria functions).

Write a test that asks the agent "Which hour of the day has the highest average fare amount?" and evaluates the response with these criteria:

* the SQL query correctly calculates average fare by hour of day
* the result identifies a specific hour as having the highest average fare
* the result includes the actual average fare amount

 

Which hour of the day has the highest average fare amount?

Implementation logic
With an LLM judge:
* a second LLM evaluates the agent's output against natural language criteria
* judge returns pass/fail with reasoning
* catches semantic failures: agent could return a valid non-null string that's completely wrong

In [3]:
%%bash
cd /Users/wesley/workspace/learning/ai-engineering-buildcamp

uv run pytest 04-testing/homework/tests/test_judge.py::test_agent_performance -v -s

============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.3, pluggy-1.6.0 -- /Users/wesley/workspace/learning/ai-engineering-buildcamp/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wesley/workspace/learning/ai-engineering-buildcamp
configfile: pyproject.toml
plugins: asyncio-1.3.0, logfire-4.32.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 1 item

04-testing/homework/tests/test_judge.py::test_agent_performance Loaded 2964624 rows
USER PROMPT (query): Which hour of the day has the highest average fare amount
TOOL CALL (query): get_schema({})
TOOL CALL (query): run_sql({"query":"SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) as hour, AVG(fare_amount) as average_fare\nFROM trips\nGROUP BY hour\nORDER BY average_fare DESC\nLIMIT 1;"})
Evaluate the agent's performance based on the following criteri

## Q6. Writing more tests

Here are more questions you can ask the agent:
* "What is the average tip amount for credit card payments?"
* "Which pickup location (PULocationID) has the most trips?"
* "What is the average fare for trips longer than 10 miles?"
* "How many trips had zero passengers recorded?"
* "What is the busiest day of the week for taxi trips?"

 

For each question, think about what scenario you are testing:
* Which columns should appear in the SQL query?
* In what order should the tools be called?
* What specific value should the result contain?

 

Write or generate tests with AI for each of these queries.

For the question "How many trips had zero passengers recorded?", which column should the agent's SQL query filter on?

In [3]:
%%bash
cd /Users/wesley/workspace/learning/ai-engineering-buildcamp

uv run pytest 04-testing/homework/tests/test_judge.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.3, pluggy-1.6.0 -- /Users/wesley/workspace/learning/ai-engineering-buildcamp/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wesley/workspace/learning/ai-engineering-buildcamp
configfile: pyproject.toml
plugins: asyncio-1.3.0, logfire-4.32.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 6 items

04-testing/homework/tests/test_judge.py::test_agent_performance SKIPPED  [ 16%]
04-testing/homework/tests/test_judge.py::test_zero_passenger_trip_count PASSED [ 33%]
04-testing/homework/tests/test_judge.py::test_average_tip_credit_card PASSED [ 50%]
04-testing/homework/tests/test_judge.py::test_busiest_pickup_location PASSED [ 66%]
04-testing/homework/tests/test_judge.py::test_average_fare_long_trips PASSED [ 83%]
04-testing/homework/tests/test_judg

In [5]:
%%bash
cd /Users/wesley/workspace/learning/ai-engineering-buildcamp

uv run pytest 04-testing/homework/tests/test_judge.py::test_zero_passenger_trip_count -v -s

============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.3, pluggy-1.6.0 -- /Users/wesley/workspace/learning/ai-engineering-buildcamp/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wesley/workspace/learning/ai-engineering-buildcamp
configfile: pyproject.toml
plugins: asyncio-1.3.0, logfire-4.32.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 1 item

04-testing/homework/tests/test_judge.py::test_zero_passenger_trip_count Loaded 2964624 rows
USER PROMPT (query): How many trips had zero passengers recorded?
TOOL CALL (query): get_schema({})
TOOL CALL (query): run_sql({"query":"SELECT COUNT(*) as zero_passengers_count FROM trips WHERE passenger_count = 0;"})
Evaluate the agent's performance based on the following criteria:
<CRITERIA>
the SQL query filters trips where passenger_count = 0 and counts t

## Q7. Cost Tracking

Add cost tracking from the module (patch_agent.py and conftest.py) and run the full test suite. What is the approximate total cost?

In [2]:
%%bash
cd /Users/wesley/workspace/learning/ai-engineering-buildcamp

uv run pytest 04-testing/homework/tests/ -v

============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.3, pluggy-1.6.0 -- /Users/wesley/workspace/learning/ai-engineering-buildcamp/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wesley/workspace/learning/ai-engineering-buildcamp
configfile: pyproject.toml
plugins: asyncio-1.3.0, logfire-4.32.1, anyio-4.13.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 8 items

04-testing/homework/tests/test_agent.py::test_agent_runs PASSED          [ 12%]
04-testing/homework/tests/test_agent.py::test_agent_tool_call_order PASSED [ 25%]
04-testing/homework/tests/test_judge.py::test_agent_performance PASSED   [ 37%]
04-testing/homework/tests/test_judge.py::test_zero_passenger_trip_count PASSED [ 50%]
04-testing/homework/tests/test_judge.py::test_average_tip_credit_card PASSED [ 62%]
04-testing/homework/tests/test_judge.py::

## Misc. Testing the database functions

In [12]:
import duckdb

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
PARQUET_FILE = "yellow_tripdata_2024-01.parquet"

## read_only=True allows DuckDB to have multiple read-only connections simultaneously so your terminal tests won't block the notebook
con = duckdb.connect("taxi.db", read_only=True)

# Fetch schema query
print("Fetch DB Schema")
print(con.execute("DESCRIBE trips").fetchall())
print()
# Fetch first 50 rows query
print("Fetch first 50 rows")
print(con.execute("SELECT * FROM trips LIMIT 50").fetchall())

Fetch DB Schema
[('VendorID', 'INTEGER', 'YES', None, None, None), ('tpep_pickup_datetime', 'TIMESTAMP', 'YES', None, None, None), ('tpep_dropoff_datetime', 'TIMESTAMP', 'YES', None, None, None), ('passenger_count', 'BIGINT', 'YES', None, None, None), ('trip_distance', 'DOUBLE', 'YES', None, None, None), ('RatecodeID', 'BIGINT', 'YES', None, None, None), ('store_and_fwd_flag', 'VARCHAR', 'YES', None, None, None), ('PULocationID', 'INTEGER', 'YES', None, None, None), ('DOLocationID', 'INTEGER', 'YES', None, None, None), ('payment_type', 'BIGINT', 'YES', None, None, None), ('fare_amount', 'DOUBLE', 'YES', None, None, None), ('extra', 'DOUBLE', 'YES', None, None, None), ('mta_tax', 'DOUBLE', 'YES', None, None, None), ('tip_amount', 'DOUBLE', 'YES', None, None, None), ('tolls_amount', 'DOUBLE', 'YES', None, None, None), ('improvement_surcharge', 'DOUBLE', 'YES', None, None, None), ('total_amount', 'DOUBLE', 'YES', None, None, None), ('congestion_surcharge', 'DOUBLE', 'YES', None, None, Non

In [7]:
con.execute("SELECT AVG(trip_distance) AS average_trip_distance FROM trips WHERE passenger_count = 2;").fetchall()

[(3.7827640377879233,)]

In [8]:
col_names = [col_info[0] for col_info in con.execute("DESCRIBE trips").fetchall()]

In [9]:
con.execute("SELECT COUNT(*) FROM trips WHERE passenger_count > 5;").fetchall()

[(22413,)]

In [10]:
con.execute("SELECT COUNT(*) FROM trips WHERE passenger_count == 0;").fetchall()

[(31465,)]